# 05 - Evaluation: Models Evaluation


In this notebook, we perform an analysis of all trained models using:
- Metric distribution analysis (plots and summary statistics)

We do not perform comparison among models, we just evaluate each single model as it is.


## Import libraries and set the paths

In [ ]:
from __future__ import annotations

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

from myocardial_infarction_mortality.config import MODELS_DIR, FIGURES_DIR
from myocardial_infarction_mortality.evaluation.visual_evaluation import plot_learning_curves, analyze_generalization_gap, analyze_roc_stability, analyze_partition_stability

In [ ]:
EXPERIMENT_NAME = "smoteenn_auto__mec_fp1_fn10"

In [ ]:
models_results_path: Path = MODELS_DIR / EXPERIMENT_NAME
print(f"Loading results at path:\n\t{models_results_path}")

In [ ]:
FIGURES_MODELS_COMPARISON_DIR = FIGURES_DIR / "EV_models_comparison_evaluation"
FIGURES_MODELS_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_MODELS_COMPARISON_EXPERIMENT_DIR = FIGURES_MODELS_COMPARISON_DIR / EXPERIMENT_NAME
FIGURES_MODELS_COMPARISON_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_SINGLE_MODEL_EVALUATION_DIR = FIGURES_MODELS_COMPARISON_EXPERIMENT_DIR / "single_models_evaluation"
FIGURES_SINGLE_MODEL_EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
pd.set_option("display.max_columns", None)
plt.rcParams.update({"font.size": 16})
sns.set_style("whitegrid")
sns.set_palette("tab10")

## Data Loading and Basic Overview

In [ ]:
files = list(models_results_path.glob("*/resubstitution_metrics_summary.csv"))
resubstitution_df = pd.DataFrame()

print(f"Found {len(files)} files. Loading...")

if files:
    # Read and Concatenate
    # We use a generator expression inside concat for memory efficiency
    resubstitution_df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

    print("Success! Combined dataframe shape:", resubstitution_df.shape)
else:
    print(f"No files found in {models_results_path.absolute()}")

In [ ]:
resubstitution_df

In [ ]:
files = list(models_results_path.glob("*/generalization_metrics_summary.csv"))
generalization_df = pd.DataFrame()

print(f"Found {len(files)} files. Loading...")

if files:
    # Read and Concatenate
    # We use a generator expression inside concat for memory efficiency
    generalization_df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

    print("Success! Combined dataframe shape:", generalization_df.shape)
else:
    print(f"No files found in {models_results_path.absolute()}")

In [ ]:
generalization_df

In [ ]:
df_results = pd.concat([resubstitution_df, generalization_df], ignore_index=True)
df_results.head(10)

In [ ]:
df_results.tail(10)

## Fix the metrics and models to evaluate and compare

In [ ]:
metrics_to_analyze = [
    "average_cost",
    "accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "roc_auc",
    "average_precision"
]
print(f"Selected metrics:\n\t{metrics_to_analyze}")

In [ ]:
models_to_analyze = df_results["model"].unique()
print(f"Selected models:\n\t{models_to_analyze}")

## Learning Stability (Train vs. Test Learning Curves)

### Objective
To analyze the stability of the model's performance across the 10 distinct random initializations (Iterations). This visualization helps determine if the model's performance is consistent regardless of the random seed or if it fluctuates wildly.

### Methodology
We aggregate the performance over the 10 folds within each iteration to compute a robust estimate for that specific random seed.
* **X-Axis:** The Iteration Number (1 to 10).
* **Y-Axis:** The Metric Score.
* **Solid Lines:** The **Mean** score across the 10 folds for that iteration.
* **Shaded Areas:** The **Standard Deviation** ($\pm 1 \sigma$) across the 10 folds, representing the intra-iteration variance.

### Interpretation Guide
1.  **Parallel Trajectories:**
    * *Ideal Scenario:* The Train (Resubstitution) and Test (Generalization) lines move in parallel. This indicates consistent generalization behavior.
2.  **Wide Shaded Bands:**
    * *Warning Sign:* Large shaded areas indicate high variance between folds within a single iteration. The model is highly sensitive to the specific data partition (e.g., a "lucky" or "unlucky" fold).
3.  **Divergence:**
    * *Overfitting Sign:* If the Train line stays high and flat while the Test line dips significantly in specific iterations, those seeds produced non-generalizable models.

In [ ]:
for model in models_to_analyze:

    print(f"Processing architecture: {model}...")

    model_save_path = FIGURES_SINGLE_MODEL_EVALUATION_DIR / model
    model_save_path.mkdir(parents=True, exist_ok=True)

    for metric in metrics_to_analyze:
        # Filter Data: Select ONLY rows belonging to this architecture
        subset_df = df_results[df_results["model"] == model].copy()

        plot_learning_curves(df=df_results,
                             metric_name=metric,
                             model_name=model,
                             save_path=model_save_path)

## Overfitting & Generalization Gap Assessment
### Objective
To rigorously assess the model's ability to generalize to unseen data by quantifying the performance drop between the training phase (Resubstitution) and the testing phase (Generalization).

### Methodology: The Generalization Gap
For each of the 100 data partitions (10 Iterations × 10 Folds), we calculate the **Generalization Gap** ($\delta$) for every metric:

$$\delta_{metric} = \text{Score}_{train} - \text{Score}_{test}$$

* **$\delta \approx 0$:** Indicates a robust model that learns generalizable patterns (Ideal).
* **$\delta > 0$:** Indicates **Overfitting**. The model is memorizing the training data and failing to perform as well on new data.
* **$\delta < 0$:** Indicates **Underfitting** or a non-representative easy test split.

### Visualization Strategy
We utilize a hybrid **Boxplot + Strip Plot** to visualize these gaps:
1.  **Boxplot:** Displays the statistical summary (Median, IQR) of the gap distribution, providing a quick view of the "average" overfitting severity.
2.  **Strip Plot:** Overlays the raw data points (100 folds) as jittered dots. This reveals the density and detects specific "outlier folds" where the model may have failed significantly.
3.  **Reference Line:** A red dashed line at $y=0$ serves as the baseline for perfect generalization.

### Interpretation Guide
* **Tight Cluster at 0:** The model is stable and trustworthy.
* **Large Positive Spread:** The model is highly sensitive to the specific data split and likely over-parameterized.
* **High Outliers:** Individual dots floating high above the boxplot indicate specific partitions where the model failed to generalize, suggesting potential data quality issues in those specific folds.

In [ ]:
for model in models_to_analyze:
    model_save_path = FIGURES_SINGLE_MODEL_EVALUATION_DIR / model
    model_save_path.mkdir(parents=True, exist_ok=True)

    # Filter Data: Select ONLY rows belonging to this architecture
    subset_df = df_results[df_results["model"] == model].copy()

    analyze_generalization_gap(df=subset_df,
                               metrics_list= metrics_to_analyze,
                               model_name=model,
                               save_path=model_save_path)

## ROC Space Stability (The "Cloud" Check)

### Objective
To assess the reliability and consistency of the model by visualizing its "operating point" stability on unseen data (Generalization split). Unlike scalar metrics (e.g., Accuracy), this analysis reveals how the model balances Sensitivity vs. Specificity across different data partitions.

### Methodology
For each of the 100 data partitions (10 Iterations $\times$ 10 Folds), we map the model's performance into the 2D ROC Space:
* **X-Axis:** False Positive Rate ($\text{FPR} = \frac{\text{FP}}{\text{FP} + \text{TN}}$) or $1 - \text{Specificity}$.
* **Y-Axis:** True Positive Rate ($\text{TPR} = \frac{\text{TP}}{\text{TP} + \text{FN}}$) or $\text{Recall}$.

### Visualization Strategy
1.  **Scatter Cloud:** We plot 100 distinct points (blue dots). A tight cloud indicates a stable model, while a dispersed cloud indicates high variance.
2.  **Centroid ($\star$):** A large red star represents the average operating point of the model across all folds.
3.  **Reference Lines:** * **Diagonal (Grey Dashed):** Represents a random classifier ($\text{AUC} = 0.5$).
    * **Ideal Point (Green Cross):** The top-left corner $(0, 1)$ representing perfect prediction.

### Interpretation Guide
By observing the shape and position of the "Cloud", we can diagnose specific stability issues:

* **Tight Cluster ("Bullet Hole"):**
    * *Verdict:* **Reliable.**
    * *Meaning:* The model is extremely stable. It makes the same trade-offs regardless of how the data is split.

* **Diagonal Streak:**
    * *Verdict:* **Unstable Thresholding.**
    * *Meaning:* The model is sensitive to class balance differences in specific folds, trading off Precision for Recall unpredictably.

* **Wide Dispersion ("Shotgun Blast"):**
    * *Verdict:* **High Variance.**
    * *Meaning:* The model is highly sensitive to noise. It works well on some data splits (top-left) but fails on others (bottom-right). This often suggests the need for better regularization or feature selection.

* **Points near Diagonal:**
    * *Verdict:* **Random Guessing.**
    * *Meaning:* On these specific folds, the model failed to learn any useful patterns.

In [ ]:
for model in models_to_analyze:
    model_save_path = FIGURES_SINGLE_MODEL_EVALUATION_DIR / model
    model_save_path.mkdir(parents=True, exist_ok=True)

    analyze_roc_stability(df=df_results,
                          model_name=model,
                          save_path=model_save_path)

## Partition Stability (Heatmap Grid)

### Objective
To diagnose whether the model's performance is consistent across all data partitions or if it relies on specific "lucky" random splits. This analysis visualizes the score variability across every single Iteration and Fold.

### Methodology
We construct a **Heatmap Grid** for each metric:
* **Grid Structure:** 10 Rows (Iterations) $\times$ 10 Columns (Folds).
* **Color Intensity:** Represents the metric score (fixed range $0.0$ to $1.0$) using the **"brg"** palette.
    * **Green:** High Performance ($\approx 1.0$).
    * **Red:** Medium Performance ($\approx 0.5$).
    * **Blue:** Low Performance ($\approx 0.0$).

### Interpretation Guide
By scanning the grid patterns, we can detect three types of behavior:

1.  **Uniform Green:**
    * *Verdict:* **Robust & High Performing.**
    * *Meaning:* The model consistently achieves high scores regardless of how the data is sliced.

2.  **"TV Static" (Random Variation):**
    * *Verdict:* **Normal Variance.**
    * *Meaning:* Slight color variations (shades of green or light green) are expected, provided there are no deep red or blue patches.

3.  **Blue/Red Stripes (Rows or Columns):**
    * *Verdict:* **Suspicious Data Sensitivity.**
    * *Meaning:*
        * **Row Stripe:** A specific Iteration (Random Seed) created a partition where the Test set was consistently harder.
        * **Column Stripe:** A specific Fold number is problematic across iterations.

4.  **Single Blue Cell:**
    * *Verdict:* **"Black Swan" Event.**
    * *Meaning:* A specific combination of data broke the model (e.g., score dropped from 0.9 to 0.4). This warrants investigation into that specific fold's data distribution.

In [ ]:
for model in models_to_analyze:
    model_save_path = FIGURES_SINGLE_MODEL_EVALUATION_DIR / model
    model_save_path.mkdir(parents=True, exist_ok=True)

    analyze_partition_stability(df=df_results,
                                model_name=model,
                                metrics_list=metrics_to_analyze,
                                save_path=model_save_path)